# 01 Differential Binding
This notebook details processes to determine differential binding of RUNX3 and RUNX1 on day 5 shCd19 and shRunx3 samples

## 01.01 Initialize Environment
Load programs needed and move to working directory

In [2]:
# Define a docker run function to simplify running docker commands
docker_run() {
    docker run --rm -it \
        -u $(id -u):$(id -g) \
        -v /home/dalbao:/home/dalbao \
        -v /etc/timezone:/etc/timezone:ro \
        -v /etc/localtime:/etc/localtime:ro \
        -w $(pwd) \
        "$@"
}

# Define software to use:
## MACS2 for peak calling
macs2() {
    docker_run quay.io/biocontainers/macs2:2.2.7.1--py38h4a8c8d9_3 macs2 "$@"
}
macs2 --version

# Change to working directory
cd /home/dalbao/AlbaoRunx3Manuscript/cutnrun
echo Working directory: $(pwd)
echo Files:
ls

macs2 2.2.7.1
Working directory: /home/dalbao/AlbaoRunx3Manuscript/cutnrun
Files:
01_diffbind  01_diffbind.ipynb	source_data


# 01.02 Re-call Peak
Reproduce .bdg files for bdgdiff.

In [3]:
# Folder to contain callpeak data
mkdir -p 01_diffbind/callpeak

# Loop over conditions and targets to call peaks using MACS2
for condition in shCd19 shRunx3; do
    for target in Runx3 Runx1; do
        echo [$(date)] Calling peaks for ${condition} ${target}
        macs2 callpeak -t source_data/markdup/${condition}_${target}_R1.target.markdup.sorted.bam \
                        -c source_data/markdup/${condition}_IgG_R1.target.markdup.sorted.bam \
                        -f BAMPE -g 1.87e9 \
                        -n ${condition}_${target} \
                        --outdir 01_diffbind/callpeak \
                        --keep-dup all -B
    done
done

[Tue Jul 14 15:11:06 EDT 2026] Calling peaks for shCd19 Runx3
INFO  @ Tue, 14 Jul 2026 15:11:07: 
# Command line: callpeak -t source_data/markdup/shCd19_Runx3_R1.target.markdup.sorted.bam -c source_data/markdup/shCd19_IgG_R1.target.markdup.sorted.bam -f BAMPE -g 1.87e9 -n shCd19_Runx3 --outdir 01_diffbind/callpeak --keep-dup all -B
# ARGUMENTS LIST:
# name = shCd19_Runx3
# format = BAMPE
# ChIP-seq file = ['source_data/markdup/shCd19_Runx3_R1.target.markdup.sorted.bam']
# control file = ['source_data/markdup/shCd19_IgG_R1.target.markdup.sorted.bam']
# effective genome size = 1.87e+09
# band width = 300
# model fold = [5, 50]
# qvalue cutoff = 5.00e-02
# The maximum gap between significant sites is assigned as the read length/tag size.
# The minimum length of peaks is assigned as the predicted fragment length "d".
# Larger dataset will be scaled towards smaller dataset.
# Range for calculating regional lambda is: 1000 bps and 10000 bps
# Broad region calling is off
# Paired-End mode is 

Check outputs:

In [4]:
ls 01_diffbind/callpeak

shCd19_Runx1_control_lambda.bdg  shRunx3_Runx1_control_lambda.bdg
shCd19_Runx1_peaks.narrowPeak	 shRunx3_Runx1_peaks.narrowPeak
shCd19_Runx1_peaks.xls		 shRunx3_Runx1_peaks.xls
shCd19_Runx1_summits.bed	 shRunx3_Runx1_summits.bed
shCd19_Runx1_treat_pileup.bdg	 shRunx3_Runx1_treat_pileup.bdg
shCd19_Runx3_control_lambda.bdg  shRunx3_Runx3_control_lambda.bdg
shCd19_Runx3_peaks.narrowPeak	 shRunx3_Runx3_peaks.narrowPeak
shCd19_Runx3_peaks.xls		 shRunx3_Runx3_peaks.xls
shCd19_Runx3_summits.bed	 shRunx3_Runx3_summits.bed
shCd19_Runx3_treat_pileup.bdg	 shRunx3_Runx3_treat_pileup.bdg


## 01.03 Determine Depths
Extract depths:

In [5]:
grep -i "total fragments in treatment" 01_diffbind/callpeak/*.xls

01_diffbind/callpeak/shCd19_Runx1_peaks.xls:# total fragments in treatment: 5406670
01_diffbind/callpeak/shCd19_Runx3_peaks.xls:# total fragments in treatment: 5866084
01_diffbind/callpeak/shRunx3_Runx1_peaks.xls:# total fragments in treatment: 5091416
01_diffbind/callpeak/shRunx3_Runx3_peaks.xls:# total fragments in treatment: 6122213


In [6]:
grep -i "fragment size is determined" 01_diffbind/callpeak/*.xls

01_diffbind/callpeak/shCd19_Runx1_peaks.xls:# fragment size is determined as 194 bps
01_diffbind/callpeak/shCd19_Runx3_peaks.xls:# fragment size is determined as 196 bps
01_diffbind/callpeak/shRunx3_Runx1_peaks.xls:# fragment size is determined as 190 bps
01_diffbind/callpeak/shRunx3_Runx3_peaks.xls:# fragment size is determined as 197 bps


## 01.04 Call Differential Regions
Call differential regions using bdgdiff.

In [7]:
# Folder to contain bdgdiff data
mkdir -p 01_diffbind/bdgdiff

macs2 bdgdiff \
  --t1 01_diffbind/callpeak/shCd19_Runx3_treat_pileup.bdg  --c1 01_diffbind/callpeak/shCd19_Runx3_control_lambda.bdg \
  --t2 01_diffbind/callpeak/shRunx3_Runx3_treat_pileup.bdg --c2 01_diffbind/callpeak/shRunx3_Runx3_control_lambda.bdg \
  --d1 5.866084 --d2 6.122213 \
  -C 3 -l 100 -g 50 \
  --o-prefix Runx3_shCd19_vs_shRunx3 --outdir 01_diffbind/bdgdiff

macs2 bdgdiff \
  --t1 01_diffbind/callpeak/shCd19_Runx1_treat_pileup.bdg  --c1 01_diffbind/callpeak/shCd19_Runx1_control_lambda.bdg \
  --t2 01_diffbind/callpeak/shRunx3_Runx1_treat_pileup.bdg --c2 01_diffbind/callpeak/shRunx3_Runx1_control_lambda.bdg \
  --d1 5.406670 --d2 5.091416 \
  -C 3 -l 100 -g 50 \
  --o-prefix Runx1_shCd19_vs_shRunx3 --outdir 01_diffbind/bdgdiff

macs2 bdgdiff \
  --t1 01_diffbind/callpeak/shCd19_Runx3_treat_pileup.bdg  --c1 01_diffbind/callpeak/shCd19_Runx3_control_lambda.bdg \
  --t2 01_diffbind/callpeak/shRunx3_Runx3_treat_pileup.bdg --c2 01_diffbind/callpeak/shRunx3_Runx3_control_lambda.bdg \
  --d1 5.866084 --d2 6.122213 \
  -C 5 -l 100 -g 50 \
  --o-prefix Runx3_shCd19_vs_shRunx3 --outdir 01_diffbind/bdgdiff

macs2 bdgdiff \
  --t1 01_diffbind/callpeak/shCd19_Runx1_treat_pileup.bdg  --c1 01_diffbind/callpeak/shCd19_Runx1_control_lambda.bdg \
  --t2 01_diffbind/callpeak/shRunx3_Runx1_treat_pileup.bdg --c2 01_diffbind/callpeak/shRunx3_Runx1_control_lambda.bdg \
  --d1 5.406670 --d2 5.091416 \
  -C 5 -l 100 -g 50 \
  --o-prefix Runx1_shCd19_vs_shRunx3 --outdir 01_diffbind/bdgdiff

INFO  @ Tue, 14 Jul 2026 15:15:05: Read and build treatment 1 bedGraph... 
INFO  @ Tue, 14 Jul 2026 15:15:08: Read and build control 1 bedGraph... 
INFO  @ Tue, 14 Jul 2026 15:15:15: Read and build treatment 2 bedGraph... 
INFO  @ Tue, 14 Jul 2026 15:15:19: Read and build control 2 bedGraph... 
INFO  @ Tue, 14 Jul 2026 15:16:03: Write peaks... 
INFO  @ Tue, 14 Jul 2026 15:16:03: Done 
INFO  @ Tue, 14 Jul 2026 15:16:04: Read and build treatment 1 bedGraph... 
INFO  @ Tue, 14 Jul 2026 15:16:08: Read and build control 1 bedGraph... 
INFO  @ Tue, 14 Jul 2026 15:16:15: Read and build treatment 2 bedGraph... 
INFO  @ Tue, 14 Jul 2026 15:16:18: Read and build control 2 bedGraph... 
INFO  @ Tue, 14 Jul 2026 15:17:01: Write peaks... 
INFO  @ Tue, 14 Jul 2026 15:17:01: Done 
INFO  @ Tue, 14 Jul 2026 15:17:03: Read and build treatment 1 bedGraph... 
INFO  @ Tue, 14 Jul 2026 15:17:06: Read and build control 1 bedGraph... 
INFO  @ Tue, 14 Jul 2026 15:17:14: Read and build treatment 2 bedGraph... 
I

In [8]:
wc -l 01_diffbind/bdgdiff/*.bed

    1077 01_diffbind/bdgdiff/Runx1_shCd19_vs_shRunx3_c3.0_common.bed
   61240 01_diffbind/bdgdiff/Runx1_shCd19_vs_shRunx3_c3.0_cond1.bed
   64500 01_diffbind/bdgdiff/Runx1_shCd19_vs_shRunx3_c3.0_cond2.bed
     363 01_diffbind/bdgdiff/Runx1_shCd19_vs_shRunx3_c5.0_common.bed
   15898 01_diffbind/bdgdiff/Runx1_shCd19_vs_shRunx3_c5.0_cond1.bed
   27469 01_diffbind/bdgdiff/Runx1_shCd19_vs_shRunx3_c5.0_cond2.bed
    3322 01_diffbind/bdgdiff/Runx3_shCd19_vs_shRunx3_c3.0_common.bed
  120867 01_diffbind/bdgdiff/Runx3_shCd19_vs_shRunx3_c3.0_cond1.bed
   94811 01_diffbind/bdgdiff/Runx3_shCd19_vs_shRunx3_c3.0_cond2.bed
    1254 01_diffbind/bdgdiff/Runx3_shCd19_vs_shRunx3_c5.0_common.bed
   43375 01_diffbind/bdgdiff/Runx3_shCd19_vs_shRunx3_c5.0_cond1.bed
   26721 01_diffbind/bdgdiff/Runx3_shCd19_vs_shRunx3_c5.0_cond2.bed
  460897 total
